In [ ]:
import torch
from PIL import Image as PILImage, ImageDraw
from IPython.display import display
import requests
from io import BytesIO
import numpy as np  # Make sure to import numpy
from segment_anything import SamAutomaticMaskGenerator
import os

# Step 1: Download the SAM model checkpoint if not already present
sam_checkpoint = "sam_vit_h_4b8939.pth"
if not os.path.exists(sam_checkpoint):
    print(f"Downloading {sam_checkpoint}...")
    url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
    response = requests.get(url)
    with open(sam_checkpoint, "wb") as f:
        f.write(response.content)
    print("Download completed.")

# Step 2: Load the SAM model
model_type = "vit_h"
device = "cuda" if torch.cuda.is_available() else "cpu"

from segment_anything import sam_model_registry

# Initialize the SAM model
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device)

# Step 3: Load the image from a URL
source = 'https://d1gymyavdvyjgt.cloudfront.net/drive/images/uploads/headers/ws_cropper/1_0x0_790x520_0x520_flattyre.jpg'
response = requests.get(source)
image_pil = PILImage.open(BytesIO(response.content)).convert('RGB')
# Convert the image to a numpy array (as SAM expects)
image = np.array(image_pil)

# Step 4: Create a SamPredictor instance and set the image

mask_generator = SamAutomaticMaskGenerator(sam)
masks = mask_generator.generate(image)

import numpy as np

def compute_iou(mask1, mask2):
    """Compute Intersection over Union (IoU) between two binary masks."""
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    iou = intersection / union if union > 0 else 0
    return iou

def group_masks_by_overlap(masks, iou_threshold=0.5):
    """Group masks into subsets based on overlap using IoU."""
    mask_subsets = []
    used_masks = set()
    
    for i, mask_data in enumerate(masks):
        if i in used_masks:
            continue
        
        main_mask = mask_data['segmentation']
        subset = [mask_data]
        used_masks.add(i)
        
        for j, other_mask_data in enumerate(masks):
            if i == j or j in used_masks:
                continue
            
            other_mask = other_mask_data['segmentation']
            iou = compute_iou(main_mask, other_mask)
            
            if iou > iou_threshold:
                subset.append(other_mask_data)
                used_masks.add(j)
        
        mask_subsets.append(subset)
    
    return mask_subsets

# Assume you have the `masks` list from mask generation
iou_threshold = 0.5  # Define your IoU threshold for grouping
mask_subsets = group_masks_by_overlap(masks, iou_threshold)
from PIL import ImageDraw

def visualize_mask_subsets(image_pil, mask_subsets):
    """Overlay original image with bounding boxes of each subset and show the image if subset size > 1."""
    for idx, subset in enumerate(mask_subsets):
        # Only display the subset if it contains more than 1 mask
        if len(subset) > 1:
            # Create a copy of the original image for drawing
            image_copy = image_pil.copy()
            draw = ImageDraw.Draw(image_copy)
            
            # Draw bounding boxes for each mask in the subset
            for mask_data in subset:
                bbox = mask_data['bbox']  # Extract the bounding box
                # Convert the bbox from XYWH to (x1, y1, x2, y2) format for drawing
                x1, y1, w, h = bbox
                x2 = x1 + w
                y2 = y1 + h
                
                # Draw the bounding box
                draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
            
            # Display the image with bounding boxes for this subset
            print(f"Displaying Subset {idx + 1} (Number of masks: {len(subset)}):")
            display(image_copy)

# Group masks by overlap using IoU (assumed to be done already)
iou_threshold = 0.4  # Define your IoU threshold for grouping
mask_subsets = group_masks_by_overlap(masks, iou_threshold)

# Visualize the subsets where the number of masks is greater than 1
visualize_mask_subsets(image_pil, mask_subsets)